<a href="https://colab.research.google.com/github/Indirajith-jithu/ArithmeticGPT/blob/Indirajith-jithu-patch-1/SLM_MATH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#generate data

from math import perm
import itertools
import random
import torch

def generate_data():

  p_numbers = list(range(9))
  n_numbers = list(range(9))

  p_num = [f"+{i}" for i in p_numbers]
  n_num = [f"-{i}" for i in n_numbers]

  my_list = p_num + n_num


  n = len(my_list)
  r = 4

  if r is None:
      r = n

  count = perm(n, r)
  print(f"The number of permutations is: {count}")




  my_list = my_list
  all_perms = itertools.permutations(my_list, r)

  data = []
  for p in all_perms:
      context = "".join(p)
      datum = context + "=" + str(eval(context)).zfill(3) + '#'
      data.append(datum)

  print('demo data',data[0])
  print("total data",len(data))


  random.shuffle(data)
  return data
data = generate_data()

split = int(0.9 * len(data))
train_text = data[:split]
val_text = data[split:]

print("train size", len(train_text), "test size", len(val_text))

chars = sorted(list(set("".join(data))))

vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

vocab_size = len(stoi)
max_seq_len = len(data[0])
print("vocab", stoi)
print("vocab size", vocab_size)
print("max seq len", max_seq_len)
def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return "".join([itos[i] for i in l])

def encode_dataset(dataset):
    return [torch.tensor(encode(s), dtype=torch.long) for s in dataset]

train_data = encode_dataset(train_text)
val_data = encode_dataset(val_text)


original_data = data[0]
print("orginal data", original_data)
encoded_data = encode(original_data)
print("encoded data", encoded_data)

print("decode data", decode(encoded_data))

The number of permutations is: 73440
demo data +0+1+2+3=006#
total data 73440
train size 66096 test size 7344
vocab {'#': 0, '+': 1, '-': 2, '0': 3, '1': 4, '2': 5, '3': 6, '4': 7, '5': 8, '6': 9, '7': 10, '8': 11, '9': 12, '=': 13}
vocab size 14
max seq len 13
orginal data +7-2+1-3=003#
encoded data [1, 10, 2, 5, 1, 4, 2, 6, 13, 3, 3, 6, 0]
decode data +7-2+1-3=003#


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

def get_batch(split, batch_size):
    data = train_data if split == "train" else val_data
    batch = random.sample(data, batch_size)

    x = torch.stack([item[:-1] for item in batch])
    y = torch.stack([item[1:] for item in batch])

    return x.to(device), y.to(device)

In [4]:
get_batch("", 8)

(tensor([[ 2,  5,  2,  8,  1,  4,  2, 11, 13,  2,  4,  7],
         [ 2,  5,  2,  9,  1,  5,  1, 11, 13,  3,  3,  5],
         [ 2,  5,  1,  7,  1,  6,  1,  5, 13,  3,  3, 10],
         [ 2,  4,  2,  8,  2,  3,  1,  7, 13,  2,  3,  5],
         [ 2,  3,  2,  6,  1, 11,  2,  7, 13,  3,  3,  4],
         [ 2, 10,  2,  9,  2,  3,  2, 11, 13,  2,  5,  4],
         [ 1,  9,  2,  5,  2,  3,  2,  6, 13,  3,  3,  4],
         [ 2, 10,  1,  7,  1, 11,  1,  8, 13,  3,  4,  3]]),
 tensor([[ 5,  2,  8,  1,  4,  2, 11, 13,  2,  4,  7,  0],
         [ 5,  2,  9,  1,  5,  1, 11, 13,  3,  3,  5,  0],
         [ 5,  1,  7,  1,  6,  1,  5, 13,  3,  3, 10,  0],
         [ 4,  2,  8,  2,  3,  1,  7, 13,  2,  3,  5,  0],
         [ 3,  2,  6,  1, 11,  2,  7, 13,  3,  3,  4,  0],
         [10,  2,  9,  2,  3,  2, 11, 13,  2,  5,  4,  0],
         [ 9,  2,  5,  2,  3,  2,  6, 13,  3,  3,  4,  0],
         [10,  1,  7,  1, 11,  1,  8, 13,  3,  4,  3,  0]]))

In [5]:
# attention

In [6]:
B, T, C = 4, 8, 32
head_size = 16
x = torch.randn((B, T, C))
x.shape


torch.Size([4, 8, 32])

In [7]:

query = nn.Linear(C, head_size, bias=False)
key = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

In [8]:
q = query(x)
k = key(x)
v = value(x)

wi = q @ k.transpose(-2, -1) / k.shape[-1] ** 2   # B, T, T

mask = torch.tril(torch.ones(T, T))
wi = wi.masked_fill(mask == 0, float("-inf"))
wi = F.softmax(wi, dim=-1)

wi = wi @ v



In [9]:


class SelfAttention(nn.Module):
  def __init__(self, emb_size, max_seq_len):
    super().__init__()
    self.query = nn.Linear(emb_size, emb_size, bias=False)
    self.key = nn.Linear(emb_size, emb_size, bias=False)
    self.value = nn.Linear(emb_size, emb_size, bias=False)

    self.register_buffer(
            "tril", torch.tril(torch.ones(max_seq_len, max_seq_len))
        )

    self.proj = nn.Linear(emb_size, emb_size)

  def forward(self, x):
    B, T, C = x.shape
    q = self.query(x)
    k = self.key(x)
    v = self.value(x)

    wi = q @ k.transpose(-2, -1) / k.shape[-1] ** 2   # B, T, T

    wi = wi.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
    wi = F.softmax(wi, dim=-1)

    out = wi @ v

    return self.proj(out)


class MultiHeadAttention(nn.Module):
  def __init__(self, emb_size, max_seq_len, head_num):
    super().__init__()

    assert emb_size % head_num == 0
    self.head_num = head_num
    self.head_dim = emb_size // head_num
    self.query = nn.Linear(emb_size, emb_size, bias=False)
    self.key = nn.Linear(emb_size, emb_size, bias=False)
    self.value = nn.Linear(emb_size, emb_size, bias=False)

    self.register_buffer(
            "mask", torch.tril(torch.ones(max_seq_len, max_seq_len))
        )
    self.proj = nn.Linear(emb_size, emb_size)



  def forward(self, x):
    B, T, C = x.shape

    q = self.query(x)
    k = self.key(x)
    v = self.value(x)

    q = q.view(B, T, self.head_num, self.head_dim)
    k = k.view(B, T, self.head_num, self.head_dim)
    v = v.view(B, T, self.head_num, self.head_dim)

    q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2) #(B, num_heads, T, head_dim)

    wi = q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5)# (B, heads, T, T)

    wi = wi.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
    wi = F.softmax(wi, dim=-1)

    out = wi @ v

    out = out.transpose(1, 2)
    out = out.contiguous().view(B, T, C)
    return self.proj(out)



test_satt = SelfAttention(C, T)

B, T, C = 4, 8, 32
test_matt = MultiHeadAttention(C, 10, T)

x = torch.randn((B, T, C))

print(x.shape)

out = test_satt(x)
out2 = test_matt(x)
out.shape, out2.shape

torch.Size([4, 8, 32])


(torch.Size([4, 8, 32]), torch.Size([4, 8, 32]))

In [10]:

class FForward(nn.Module):

  def __init__(self, n_emb, exp, dropout=0.5):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_emb, exp * n_emb),
        nn.ReLU(),
        nn.Linear(exp*n_emb, n_emb),
        nn.Dropout(dropout)

    )
  def forward(self, x):
    return self.net(x)

In [11]:
class Block(nn.Module):
  def __init__(self, n_emb, max_seq_len, attention, exp, dropout, head_size=None):
    super().__init__()
    self.att = SelfAttention(n_emb, max_seq_len) \
                      if attention == "self" \
                    else MultiHeadAttention(n_emb, max_seq_len, head_size)

    self.ffwd = FForward(n_emb, exp, dropout)

    self.att_norm = nn.LayerNorm(n_emb)
    self.ffws_norm = nn.LayerNorm(n_emb)

  def forward(self, x):
    x = x + self.att_norm(self.att(x))
    x = x + self.ffws_norm(self.ffwd(x))
    return x





In [12]:


class MathGPT(nn.Module):

  def __init__(self, vocab_size, n_emb, max_seq_len, n_block, attention, exp, dropout, head_size=None):
    super().__init__()

    self.token_emb = nn.Embedding(vocab_size, n_emb)
    self.position_embedding_table = nn.Embedding(max_seq_len, n_emb)

    self.blocks = nn.ModuleList(
        Block(n_emb, max_seq_len, attention, exp, dropout, head_size) for _ in range(n_block)
    )

    self.lm_head = nn.Linear(n_emb, vocab_size)

    self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

  def forward(self, idx, target=None):

    B, T = idx.shape

    t_embd = self.token_emb(idx)
    p_embd = self.position_embedding_table(
        torch.arange(T, device=idx.device)
    )

    x = t_embd + p_embd

    for block in self.blocks:
      x = block(x)

    logits = self.lm_head(x)

    loss = None

    if target is not None:
      loss = self.loss_fn(
          logits.reshape(-1, vocab_size),
          target.reshape(-1)
      )
    return logits, loss






In [161]:
# utils
def validation(model):
  with torch.no_grad():
    n_batch = 6000
    xv, yv = get_batch("test", n_batch)
    batchin = xv[:,:-3]
    for _ in range(3):
      put, _ = model(batchin)
      max_ = put[:,-1:,:].argmax(dim=-1)
      batchin = torch.concat([batchin, max_], dim=-1)
    cmp = torch.eq(batchin, xv)
    rcmp = torch.all(cmp, dim=-1)
    return rcmp.sum().item() / n_batch

In [191]:
vocab_size = vocab_size
max_seq_len = max_seq_len
n_emb = 16
n_block = 4
attention = "self-"
exp = 4
dropout = 0.3
head_size=4

model = MathGPT(vocab_size, n_emb, max_seq_len + 1, n_block, attention, exp, dropout, head_size).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")


Total parameters: 13,614


In [192]:
# train loop

In [193]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

In [ ]:
max_iters = 42000
eval_interval = 550
batch_size = 128
model.train()
for step in range(max_iters):

    xb, yb = get_batch("train", batch_size)

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # break
    if step % eval_interval == 0:
        print("train loss:", loss.item(), end=" - ")
        model.eval()
        val_acc = validation(model)
        print("val acc:", val_acc)
        model.train()
        # print("-" * 30)

print("Training Complete.")

train loss: 1.1408135890960693 - val acc: 0.105
train loss: 1.1243788003921509 - val acc: 0.1645
train loss: 1.1041263341903687 - val acc: 0.22966666666666666
train loss: 1.0727283954620361 - val acc: 0.31066666666666665
train loss: 1.0765460729599 - val acc: 0.3155
train loss: 1.0614409446716309 - val acc: 0.34883333333333333
train loss: 1.04072105884552 - val acc: 0.4265
train loss: 1.029662013053894 - val acc: 0.5323333333333333
train loss: 1.0196675062179565 - val acc: 0.44483333333333336
train loss: 1.003064751625061 - val acc: 0.48
train loss: 1.0120363235473633 - val acc: 0.5586666666666666
train loss: 0.9837353825569153 - val acc: 0.6245
train loss: 0.9904566407203674 - val acc: 0.7355
train loss: 0.9838975071907043 - val acc: 0.7105
train loss: 0.9830247759819031 - val acc: 0.788
train loss: 0.9750681519508362 - val acc: 0.7343333333333333
train loss: 0.9724982380867004 - val acc: 0.7338333333333333
train loss: 0.9622706770896912 - val acc: 0.8956666666666667
train loss: 0.955

Total parameters: 18,254


In [30]:
def generate(idx):
  model.eval()
  tokens = idx
  while len(tokens) <= 12:
    out, _ = model(tokens.unsqueeze(0))
    idx_next = out[:, -1, :].argmax(keepdim=True)[0]
    tokens = torch.cat([tokens, idx_next])
  return tokens



In [51]:
test_data = xb[9]
print(decode(test_data.tolist()))
test_data = test_data[:-3]
decode(test_data.tolist())

+4+6+8-0=018


'+4+6+8-0='

In [54]:
put = generate(test_data)

In [55]:
decode(put.tolist())

'+4+6+8-0=018#'

In [ ]:
put

tensor([ 2, 10,  1,  8,  2,  9,  1,  9, 13,  2,  3,  5,  0])

In [ ]:
xb[8]

tensor([ 2, 10,  1,  8,  2,  9,  1,  9, 13,  2,  3,  5])

In [150]:
decode(xv[0].tolist())

'+2-0-2+7=007'

In [151]:

# batchin

In [153]:

# max_

In [154]:

# batchin

In [155]:
xv[:,:]

tensor([[ 1,  5,  2,  ...,  3,  3, 10],
        [ 2,  8,  2,  ...,  2,  4,  7],
        [ 1, 10,  1,  ...,  3,  3,  7],
        ...,
        [ 1,  3,  1,  ...,  3,  3,  4],
        [ 1, 10,  2,  ...,  3,  4,  5],
        [ 2,  6,  2,  ...,  3,  3, 10]])

tensor(4879)

In [159]:
validation()

0.8166666666666667